In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from scipy.stats import poisson

data = pd.read_csv("Data/results.csv")

wc_countries = ["Canada", "Mexico", "United States", "Algeria", "Argentina", "Australia", "Austria", "Belgium", "Bosnia and Herzegovina", "Brazil", "Cape Verde", "Colombia", "DR Congo", "Croatia", "Curaçao", "Czech Republic", "Ecuador", "Egypt", "England", "France", "Germany", "Ghana", "Haiti", "Iraq", "Iran", "Ivory Coast", "Japan", "Jordan", "Korea Republic", "Morocco", "Netherlands", "New Zealand", "Norway", "Panama", "Paraguay", "Peru", "Qatar", "Saudi Arabia", "Scotland", "Senegal", "South Africa","Spain","Sweden","Switzerland","Tunisia","Turkey","Uruguay","Uzbekistan"]


In [2]:
team_to_i= {team: i for i, team in enumerate(wc_countries)}

data["home_team"] = data["home_team"].map(team_to_i)
data["away_team"] = data["away_team"].map(team_to_i)

# Keep only matches between tracked World Cup teams and with valid scores.
data = data.dropna(subset=["home_team", "away_team", "home_score", "away_score"]).copy()
data["home_team"] = data["home_team"].astype(int)
data["away_team"] = data["away_team"].astype(int)

data['home_adv_multiplier'] = np.where(data['neutral'] == True, 0, 1)

In [3]:
def dixon_coles_nll(params, df, n):
    # Unpack the current guessed parameters
    attack = params[:n]
    defense = params[n:2*n]
    home_adv = params[-2]
    rho = params[-1]

    # Use the mapped integer team indices stored in 'home_team' and 'away_team'
    home_idxs = df['home_team'].values
    away_idxs = df['away_team'].values

    # Calculate xG for every match in the dataset based on these parameters
    # Notice we multiply home_adv by the multiplier (0 if neutral, 1 if home stadium)
    home_xg = np.exp(attack[home_idxs] + 
                     defense[away_idxs] + 
                     (home_adv * df['home_adv_multiplier'].values))

    away_xg = np.exp(attack[away_idxs] + 
                     defense[home_idxs])

    # Get the actual goals scored
    x = df['home_score'].values
    y = df['away_score'].values

    # Calculate standard Poisson Log-Likelihood
    ll_home = poisson.logpmf(x, home_xg)
    ll_away = poisson.logpmf(y, away_xg)

    # Apply the Dixon-Coles Rho Adjustment
    tau = np.ones(len(df))

    mask_00 = (x == 0) & (y == 0)
    tau[mask_00] = 1 - (home_xg[mask_00] * away_xg[mask_00] * rho)

    mask_01 = (x == 0) & (y == 1)
    tau[mask_01] = 1 + (home_xg[mask_01] * rho)

    mask_10 = (x == 1) & (y == 0)
    tau[mask_10] = 1 + (away_xg[mask_10] * rho)

    mask_11 = (x == 1) & (y == 1)
    tau[mask_11] = 1 - rho

    # Prevent negative values from crashing the log function
    tau = np.maximum(tau, 1e-10)

    # Return the Negative Log-Likelihood (the penalty score)
    return -np.sum(ll_home + ll_away + np.log(tau))

In [4]:
# Number of World Cup Teams
n = len(wc_countries)

## 1. Setup Initial Guesses (1.0 for stats, small non-zero modifiers)
initial_guess = np.concatenate([
    np.ones(n),  # Attack
    np.ones(n),  # Defense
    np.array([0.1]),   # Home Advantage
    np.array([0.05])   # Rho
])

# 2. Setup Bounds to keep math realistic
bounds = [(0.01, 5.0)] * (2 * n) + [(0.0, 2.0)] + [(-0.3, 0.3)]

# 3. Setup Constraint (Sum of attacks must equal number of teams)
def constraint_func(params):
    return sum(params[:n]) - n

constraints = [{'type': 'eq', 'fun': constraint_func}]

# 4. Train the Model!
print("Training model... this may take a moment.")
result = minimize(
    dixon_coles_nll,
    initial_guess,
    args=(data, n),
    method='SLSQP',
    bounds=bounds,
    constraints=constraints,
    options={'maxiter': 200}
)

print('Optimization success:', result.success)
print('Optimization message:', result.message)

# 5. Extract the optimized parameters into usable variables
opt_attack = result.x[:n]
opt_defense = result.x[n:2*n]
opt_home_adv = result.x[-2]
opt_rho = result.x[-1]

print(f"Model Trained! Optimal Rho: {opt_rho:.4f}")

Training model... this may take a moment.
Optimization success: True
Optimization message: Optimization terminated successfully
Model Trained! Optimal Rho: -0.0878


In [6]:
def predict_match(home_team, away_team, is_neutral=False, max_goals=6):
    # 1. Look up the numerical IDs for the teams
    home_idx = team_to_i[home_team]
    away_idx = team_to_i[away_team]

    # 2. Apply Home Advantage (or 0 if neutral)
    h_adv = 0 if is_neutral else opt_home_adv

    # 3. Calculate Expected Goals (xG) using our optimized parameters
    home_xg = np.exp(opt_attack[home_idx] + opt_defense[away_idx] + h_adv)
    away_xg = np.exp(opt_attack[away_idx] + opt_defense[home_idx])

    # 4. Generate the Probability Matrix
    matrix = np.zeros((max_goals, max_goals))
    for x in range(max_goals):
        for y in range(max_goals):
            matrix[x, y] = poisson.pmf(x, home_xg) * poisson.pmf(y, away_xg)

    # 5. Apply the Rho adjustment to the 4 core scorelines
    matrix[0, 0] *= max(0, 1 - (home_xg * away_xg * opt_rho))
    matrix[0, 1] *= max(0, 1 + (home_xg * opt_rho))
    matrix[1, 0] *= max(0, 1 + (away_xg * opt_rho))
    matrix[1, 1] *= max(0, 1 - opt_rho)

    # 6. Calculate Match Odds
    home_win = np.tril(matrix, -1).sum()
    draw = np.trace(matrix)
    away_win = np.triu(matrix, 1).sum()

    print(f"--- {home_team} vs {away_team} ---")
    print(f"Home Win: {home_win:.2%}")
    print(f"Draw: {draw:.2%}")
    print(f"Away Win: {away_win:.2%}")
    
    return matrix

# Example: Predict Canada vs USA (Neutral Venue)
match_matrix = predict_match('Canada', 'United States', is_neutral=True)

# 2. Convert the raw NumPy array into a labeled DataFrame
# Rows are the Home Team's goals (x), Columns are the Away Team's goals (y)
df_matrix = pd.DataFrame(
    match_matrix, 
    index=[f"Home {i}" for i in range(6)], 
    columns=[f"Away {i}" for i in range(6)]
)

# 3. View in the console as readable percentages
print("\n--- Console Output ---")
print(df_matrix.map(lambda prob: f"{prob:.2%}"))

--- Canada vs United States ---
Home Win: 28.17%
Draw: 22.44%
Away Win: 46.76%

--- Console Output ---
       Away 0 Away 1 Away 2 Away 3 Away 4 Away 5
Home 0  3.21%  4.48%  5.48%  3.84%  2.01%  0.85%
Home 1  3.23%  9.05%  8.74%  6.12%  3.21%  1.35%
Home 2  3.16%  6.64%  6.97%  4.88%  2.56%  1.08%
Home 3  1.68%  3.53%  3.71%  2.59%  1.36%  0.57%
Home 4  0.67%  1.41%  1.48%  1.03%  0.54%  0.23%
Home 5  0.21%  0.45%  0.47%  0.33%  0.17%  0.07%
